In [27]:
import open_clip
from clip import clip
import torch
import numpy as np

In [3]:
url = clip._MODELS['ViT-B/16']
model_path = clip._download(url)
model = torch.jit.load(model_path, map_location="cpu").eval()

In [4]:
clip_model = clip.build_model(model.state_dict())

In [5]:
clip_model.dtype

torch.float16

In [6]:
oc_model, _, _ = open_clip.create_model_and_transforms('ViT-B-16', 'openai', device='cpu', force_quick_gelu=True)

In [7]:
oc_model.transformer.get_cast_dtype()

torch.float32

In [86]:
oc_model.dtype = torch.float32

https://discuss.pytorch.org/t/when-should-i-use-nn-modulelist-and-when-should-i-use-nn-sequential/5463/5

In [51]:
clip_model.visual

VisionTransformer(
  (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
  (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): Sequential(
      (0): ResidualAttentionBlock(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): QuickGELU()
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (1): ResidualAttentionBlock(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise

In [50]:
oc_model.visual

VisionTransformer(
  (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
  (patch_dropout): Identity()
  (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): ModuleList(
      (0-11): 12 x ResidualAttentionBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (ls_1): Identity()
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): QuickGELU()
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
        (ls_2): Identity()
      )
    )
  )
  (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [14]:
print(clip_model.positional_embedding)
print(oc_model.positional_embedding)

Parameter containing:
tensor([[-2.1003e-04, -8.2927e-05, -4.0602e-03,  ...,  4.8409e-04,
          2.7823e-03,  4.6133e-03],
        [ 2.0279e-03, -2.3657e-03, -9.2590e-04,  ...,  2.6768e-03,
          7.5780e-04,  1.6466e-04],
        [-1.7824e-03,  1.0458e-03, -7.0424e-04,  ...,  7.2783e-04,
          2.5611e-03,  4.2764e-04],
        ...,
        [ 1.8419e-04,  1.3991e-03, -8.9098e-05,  ..., -3.0847e-03,
         -4.8699e-03,  3.6575e-03],
        [-5.9546e-03,  3.7494e-03,  8.8027e-03,  ..., -1.3645e-03,
         -3.8477e-03,  6.4736e-03],
        [ 5.6348e-03, -1.7269e-02, -1.5312e-02,  ..., -6.6386e-03,
          5.5739e-03, -1.8557e-02]], requires_grad=True)
Parameter containing:
tensor([[-2.1003e-04, -8.2927e-05, -4.0602e-03,  ...,  4.8409e-04,
          2.7823e-03,  4.6133e-03],
        [ 2.0279e-03, -2.3657e-03, -9.2590e-04,  ...,  2.6768e-03,
          7.5780e-04,  1.6466e-04],
        [-1.7824e-03,  1.0458e-03, -7.0424e-04,  ...,  7.2783e-04,
          2.5611e-03,  4.2764e-

In [15]:
print(clip_model.ln_final)
print(oc_model.ln_final)

LayerNorm((512,), eps=1e-05, elementwise_affine=True)
LayerNorm((512,), eps=1e-05, elementwise_affine=True)


In [16]:
print(clip_model.text_projection)
print(oc_model.text_projection)

Parameter containing:
tensor([[-0.0087,  0.0040, -0.0137,  ...,  0.0086, -0.0194, -0.0168],
        [-0.0014,  0.0102,  0.0042,  ..., -0.0243,  0.0035,  0.0095],
        [ 0.0026, -0.0003,  0.0160,  ...,  0.0108, -0.0045, -0.0229],
        ...,
        [-0.0008,  0.0023,  0.0237,  ...,  0.0073,  0.0101, -0.0112],
        [ 0.0183, -0.0002, -0.0152,  ..., -0.0391,  0.0010,  0.0174],
        [ 0.0037, -0.0013,  0.0052,  ...,  0.0135,  0.0056, -0.0070]],
       dtype=torch.float16, requires_grad=True)
Parameter containing:
tensor([[-0.0087,  0.0040, -0.0137,  ...,  0.0086, -0.0194, -0.0168],
        [-0.0014,  0.0102,  0.0042,  ..., -0.0243,  0.0035,  0.0095],
        [ 0.0026, -0.0003,  0.0160,  ...,  0.0108, -0.0045, -0.0229],
        ...,
        [-0.0008,  0.0023,  0.0237,  ...,  0.0073,  0.0101, -0.0112],
        [ 0.0183, -0.0002, -0.0152,  ..., -0.0391,  0.0010,  0.0174],
        [ 0.0037, -0.0013,  0.0052,  ...,  0.0135,  0.0056, -0.0070]],
       requires_grad=True)


In [42]:
print(clip_model.ln_final.weight.shape[0])
print(oc_model.ln_final.weight.shape[0])

512
512


In [43]:
print(clip_model.visual.input_resolution)
print(oc_model.visual.input_resolution)

224


AttributeError: 'VisionTransformer' object has no attribute 'input_resolution'

In [45]:
print(clip_model.token_embedding)
print(oc_model.token_embedding)

Embedding(49408, 512)
Embedding(49408, 512)


In [46]:
print(clip_model.logit_scale)
print(oc_model.logit_scale)

Parameter containing:
tensor(4.6052, requires_grad=True)
Parameter containing:
tensor(4.6052, requires_grad=True)
